# 🧪 Стресс-тестирование финансовой модели

Этот ноутбук позволяет:
- Выбрать компанию и сценарий стресс-тестирования
- Применить различные типы шоков (Revenue, Margin, WC, CapEx, Rates/FX, Debt Rollover)
- Запустить модель с модифицированными драйверами
- Сравнить результаты базового и стресс-сценариев
- Просмотреть ключевые метрики и чувствительность модели

## 📋 Этапы работы:

1. **Загрузка данных**: Загрузка базовых драйверов и исторического состояния
2. **Выбор сценария**: Выбор стресс-сценария из YAML конфигурации
3. **Применение шоков**: Автоматическое применение шоков к драйверам и макро-факторам
4. **Запуск модели**: Запуск модели с модифицированными драйверами
5. **Анализ результатов**: Сравнение базового и стресс-сценариев

## 📚 Документация

- `docs/stress_testing/STRESS_TESTING_GUIDE.md` - Полное руководство
- `configs/stress_scenarios.yaml` - Конфигурация сценариев

In [ ]:
# Импорты и настройка
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / 'engine').exists():
    ROOT = ROOT.parent
if not (ROOT / 'engine').exists():
    raise RuntimeError(f"Can't locate project root from {Path.cwd()}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"📁 ROOT: {ROOT}")

In [ ]:
# Импорты модулей
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from typing import Optional, Dict, Any
from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout, Output

# Импорты движка
from engine.database.data_mart import get_data_mart
from engine.model3.io import load_inputs
from engine.stress.stress_runner import StressRunner
from engine.stress.scenarios import ScenarioLoader

print("✅ Модули загружены")

## 1️⃣ Выбор компании и сценария

In [ ]:
# Список компаний
COMPANIES = [d.name for d in (ROOT / 'companies').iterdir() 
             if d.is_dir() and not d.name.startswith('.')]

# Загрузчик сценариев
scenario_loader = ScenarioLoader()
scenario_loader.load(ROOT)
available_scenarios = scenario_loader.list_scenarios()
available_sectors = scenario_loader.list_sectors()

print(f"📊 Доступные компании: {COMPANIES}")
print(f"📋 Доступные сценарии: {available_scenarios}")
print(f"🏭 Доступные секторы: {available_sectors}")

# Виджеты для выбора
company_dropdown = widgets.Dropdown(
    options=COMPANIES,
    value='rusal' if 'rusal' in COMPANIES else (COMPANIES[0] if COMPANIES else None),
    description='Компания:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

scenario_dropdown = widgets.Dropdown(
    options=available_scenarios,
    value=available_scenarios[0] if available_scenarios else None,
    description='Сценарий:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

sector_dropdown = widgets.Dropdown(
    options=[''] + available_sectors,
    value='',
    description='Сектор (опционально):',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

forecast_years_text = widgets.Text(
    value='2025,2026,2027,2028,2029',
    description='Годы прогноза:',
    style={'description_width': '150px'},
    layout=Layout(width='400px')
)

run_stress_button = widgets.Button(
    description='🚀 Запустить стресс-тест',
    button_style='success',
    icon='play'
)

compare_scenarios_button = widgets.Button(
    description='📊 Сравнить сценарии',
    button_style='info',
    icon='chart-bar'
)

status_output = widgets.Output()
results_output = widgets.Output()

display(VBox([
    HBox([company_dropdown, scenario_dropdown]),
    HBox([sector_dropdown, forecast_years_text]),
    HBox([run_stress_button, compare_scenarios_button]),
    status_output,
    results_output
]))

## 4️⃣ Визуализация результатов

Визуализация ключевых метрик для сравнения сценариев.

In [ ]:
# Визуализация результатов стресс-тестирования
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка стиля
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def plot_stress_comparison(scenario_names: List[str], metric_name: str, years: List[int]):
    """Визуализация сравнения метрик по сценариям"""
    if not stress_results:
        print("⚠️ Нет результатов для визуализации. Сначала запустите стресс-тесты.")
        return
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for scenario_name in scenario_names:
        if scenario_name not in stress_results:
            continue
        
        result = stress_results[scenario_name]
        if not result.get("success") or not result.get("results"):
            continue
        
        is_frame = result["results"].get("income_statement")
        if is_frame is None or metric_name not in is_frame.index:
            continue
        
        values = []
        for year in years:
            if year in is_frame.columns:
                values.append(is_frame.loc[metric_name, year])
            else:
                values.append(None)
        
        ax.plot(years, values, marker='o', label=result.get('scenario_label', scenario_name), linewidth=2)
    
    ax.set_xlabel('Год', fontsize=12)
    ax.set_ylabel(metric_name.replace('_', ' ').title(), fontsize=12)
    ax.set_title(f'Сравнение {metric_name.replace("_", " ").title()} по сценариям', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.xticks(years)
    plt.tight_layout()
    plt.show()

def plot_sensitivity_analysis(scenario_names: List[str], metrics: List[str], year: int):
    """Анализ чувствительности метрик к стресс-сценариям"""
    if not stress_results:
        print("⚠️ Нет результатов для визуализации. Сначала запустите стресс-тесты.")
        return
    
    # Загружаем базовые результаты для сравнения
    company = company_dropdown.value
    forecast_years = parse_forecast_years(forecast_years_text.value)
    
    try:
        hist_state, history, drivers, canonical, _ = load_inputs(root=ROOT, company=company)
        
        # Запускаем базовую модель (без стресса)
        from engine.model3.core import ThreeStatementModel
        base_model = ThreeStatementModel(
            hist=hist_state,
            drv=drivers,
            years=forecast_years
        )
        base_model.run()
        
        base_is = base_model.get_income_statement()
        base_bs = base_model.get_balance_sheet()
        
        # Собираем данные для визуализации
        comparison_data = []
        
        # Базовый сценарий
        base_row = {"Сценарий": "Baseline", "Тип": "baseline"}
        for metric in metrics:
            if metric in base_is.index and year in base_is.columns:
                base_row[metric] = base_is.loc[metric, year]
            elif metric in base_bs.index and year in base_bs.columns:
                base_row[metric] = base_bs.loc[metric, year]
            else:
                base_row[metric] = None
        comparison_data.append(base_row)
        
        # Стресс-сценарии
        for scenario_name in scenario_names:
            if scenario_name not in stress_results:
                continue
            
            result = stress_results[scenario_name]
            if not result.get("success") or not result.get("results"):
                continue
            
            stress_row = {
                "Сценарий": result.get('scenario_label', scenario_name),
                "Тип": "stress"
            }
            
            results = result["results"]
            is_frame = results.get("income_statement")
            bs_frame = results.get("balance_sheet")
            
            for metric in metrics:
                if is_frame is not None and metric in is_frame.index and year in is_frame.columns:
                    stress_row[metric] = is_frame.loc[metric, year]
                elif bs_frame is not None and metric in bs_frame.index and year in bs_frame.columns:
                    stress_row[metric] = bs_frame.loc[metric, year]
                else:
                    stress_row[metric] = None
            
            comparison_data.append(stress_row)
        
        # Создаем DataFrame
        df = pd.DataFrame(comparison_data)
        
        # Визуализация
        fig, axes = plt.subplots(1, len(metrics), figsize=(6*len(metrics), 6))
        if len(metrics) == 1:
            axes = [axes]
        
        for idx, metric in enumerate(metrics):
            ax = axes[idx]
            
            # Базовое значение
            baseline_value = df[df["Тип"] == "baseline"][metric].iloc[0] if not df[df["Тип"] == "baseline"].empty else None
            
            # Стресс-значения
            stress_df = df[df["Тип"] == "stress"]
            if not stress_df.empty and metric in stress_df.columns:
                values = stress_df[metric].dropna()
                scenarios = stress_df["Сценарий"].tolist()
                
                # Создаем столбчатую диаграмму
                bars = ax.bar(range(len(values)), values, alpha=0.7)
                
                # Линия базового значения
                if baseline_value is not None:
                    ax.axhline(y=baseline_value, color='r', linestyle='--', linewidth=2, label='Baseline')
                
                ax.set_xticks(range(len(values)))
                ax.set_xticklabels(scenarios, rotation=45, ha='right')
                ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=10)
                ax.set_title(f'{metric.replace("_", " ").title()} - Анализ чувствительности', fontsize=12, fontweight='bold')
                ax.legend()
                ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
        
        # Таблица сравнения
        display(Markdown("### 📊 Таблица сравнения метрик"))
        display(df.set_index("Сценарий")[metrics])
        
    except Exception as e:
        print(f"Ошибка при анализе чувствительности: {e}")
        import traceback
        traceback.print_exc()

# Примеры использования:
# plot_stress_comparison(['mild', 'moderate', 'severe'], 'revenue_total', [2025, 2026, 2027, 2028, 2029])
# plot_sensitivity_analysis(['mild', 'moderate'], ['revenue_total', 'ebitda', 'net_income'], 2025)

In [ ]:
# Глобальные переменные для результатов
stress_results = {}
base_results = None

def parse_forecast_years(years_str: str) -> list:
    """Парсинг строки с годами"""
    try:
        years = [int(y.strip()) for y in years_str.split(',')]
        return sorted(years)
    except Exception as e:
        print(f"Ошибка парсинга лет: {e}")
        return [2025, 2026, 2027, 2028, 2029]

def run_stress_test(b):
    """Запуск стресс-теста для выбранного сценария"""
    global stress_results, base_results
    
    with status_output:
        status_output.clear_output()
        print("🔄 Запуск стресс-теста...")
    
    try:
        company = company_dropdown.value
        scenario_name = scenario_dropdown.value
        sector = sector_dropdown.value if sector_dropdown.value else None
        forecast_years = parse_forecast_years(forecast_years_text.value)
        
        # Загружаем базовые данные
        with status_output:
            status_output.clear_output()
            print(f"📂 Загрузка данных для компании: {company}")
        
        hist_state, history, drivers, canonical, cf_corrections = load_inputs(
            root=ROOT,
            company=company
        )
        
        # Получаем DataMart и MacroRepository
        mart = get_data_mart(ROOT, company)
        macro_repo = mart.macro_repository if hasattr(mart, 'macro_repository') else None
        
        # Создаем StressRunner
        runner = StressRunner(
            root=ROOT,
            company=company,
            macro_repo=macro_repo,
            mart=mart
        )
        
        # Запускаем стресс-сценарий
        with status_output:
            status_output.clear_output()
            print(f"🧪 Запуск сценария: {scenario_name}")
            if sector:
                print(f"🏭 Сектор: {sector}")
        
        result = runner.run_stress_scenario(
            scenario_name=scenario_name,
            base_hist=hist_state,
            base_drivers=drivers,
            forecast_years=forecast_years,
            sector=sector
        )
        
        stress_results[scenario_name] = result
        
        # Отображаем результаты
        with results_output:
            results_output.clear_output()
            
            if result["success"]:
                display(Markdown(f"## ✅ Сценарий '{result['scenario_label']}' выполнен успешно"))
                
                # Показываем ключевые метрики
                results = result["results"]
                if results:
                    is_frame = results.get("income_statement")
                    bs_frame = results.get("balance_sheet")
                    cf_frame = results.get("cash_flow")
                    
                    if is_frame is not None:
                        display(Markdown("### 📊 Income Statement (ключевые метрики)"))
                        key_metrics = ['revenue_total', 'ebitda', 'ebit', 'net_income']
                        available_metrics = [m for m in key_metrics if m in is_frame.index]
                        if available_metrics:
                            display(is_frame.loc[available_metrics])
                    
                    if bs_frame is not None:
                        display(Markdown("### 💰 Balance Sheet (ключевые метрики)"))
                        key_metrics = ['cash', 'total_debt', 'total_equity']
                        available_metrics = [m for m in key_metrics if m in bs_frame.index]
                        if available_metrics:
                            display(bs_frame.loc[available_metrics])
                    
                    if cf_frame is not None:
                        display(Markdown("### 💵 Cash Flow (ключевые метрики)"))
                        key_metrics = ['cfo', 'cfi', 'cff', 'net_change_in_cash']
                        available_metrics = [m for m in key_metrics if m in cf_frame.index]
                        if available_metrics:
                            display(cf_frame.loc[available_metrics])
            else:
                display(Markdown(f"## ❌ Ошибка при выполнении сценария"))
                display(Markdown(f"**Ошибка:** {result.get('error', 'Неизвестная ошибка')}"))
        
        with status_output:
            status_output.clear_output()
            print("✅ Стресс-тест завершен")
    
    except Exception as e:
        with status_output:
            status_output.clear_output()
            print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()

def compare_scenarios_func(b):
    """Сравнение нескольких сценариев"""
    global stress_results
    
    with status_output:
        status_output.clear_output()
        print("📊 Сравнение сценариев...")
    
    try:
        company = company_dropdown.value
        forecast_years = parse_forecast_years(forecast_years_text.value)
        sector = sector_dropdown.value if sector_dropdown.value else None
        
        # Загружаем базовые данные
        hist_state, history, drivers, canonical, cf_corrections = load_inputs(
            root=ROOT,
            company=company
        )
        
        # Получаем DataMart и MacroRepository
        mart = get_data_mart(ROOT, company)
        macro_repo = mart.macro_repository if hasattr(mart, 'macro_repository') else None
        
        # Создаем StressRunner
        runner = StressRunner(
            root=ROOT,
            company=company,
            macro_repo=macro_repo,
            mart=mart
        )
        
        # Сравниваем все доступные сценарии
        comparison = runner.compare_scenarios(
            scenario_names=available_scenarios,
            base_hist=hist_state,
            base_drivers=drivers,
            forecast_years=forecast_years,
            sector=sector
        )
        
        # Отображаем сравнение
        with results_output:
            results_output.clear_output()
            display(Markdown("## 📊 Сравнение сценариев"))
            
            # Создаем таблицу сравнения
            comparison_data = []
            for scenario_name, scenario_result in comparison["scenarios"].items():
                if scenario_result.get("success") and scenario_result.get("results"):
                    results = scenario_result["results"]
                    is_frame = results.get("income_statement")
                    
                    row = {
                        "Сценарий": scenario_result.get("scenario_label", scenario_name),
                        "Статус": "✅ Успешно"
                    }
                    
                    # Добавляем ключевые метрики для первого года
                    if is_frame is not None and forecast_years:
                        first_year = forecast_years[0]
                        if 'revenue_total' in is_frame.index and first_year in is_frame.columns:
                            row["Revenue"] = f"{is_frame.loc['revenue_total', first_year]:,.0f}"
                        if 'ebitda' in is_frame.index and first_year in is_frame.columns:
                            row["EBITDA"] = f"{is_frame.loc['ebitda', first_year]:,.0f}"
                        if 'net_income' in is_frame.index and first_year in is_frame.columns:
                            row["Net Income"] = f"{is_frame.loc['net_income', first_year]:,.0f}"
                    
                    comparison_data.append(row)
                else:
                    comparison_data.append({
                        "Сценарий": scenario_result.get("scenario_label", scenario_name),
                        "Статус": f"❌ {scenario_result.get('error', 'Ошибка')}"
                    })
            
            if comparison_data:
                comparison_df = pd.DataFrame(comparison_data)
                display(comparison_df)
        
        with status_output:
            status_output.clear_output()
            print("✅ Сравнение завершено")
    
    except Exception as e:
        with status_output:
            status_output.clear_output()
            print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()

# Привязываем обработчики
run_stress_button.on_click(run_stress_test)
compare_scenarios_button.on_click(compare_scenarios_func)

## 2️⃣ Просмотр конфигурации сценариев

In [ ]:
# Просмотр конфигурации сценариев
config_path = ROOT / "configs" / "stress_scenarios.yaml"

if config_path.exists():
    with config_path.open("r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    
    display(Markdown("### 📋 Доступные сценарии:"))
    for scenario_name, scenario_config in config.get("scenarios", {}).items():
        label = scenario_config.get("label", scenario_name)
        description = scenario_config.get("description", "")
        display(Markdown(f"- **{scenario_name}**: {label}"))
        if description:
            display(Markdown(f"  - {description}"))
    
    if config.get("sector_packs"):
        display(Markdown("### 🏭 Отраслевые пакеты:"))
        for sector_name, sector_config in config["sector_packs"].items():
            label = sector_config.get("label", sector_name)
            display(Markdown(f"- **{sector_name}**: {label}"))
else:
    display(Markdown("⚠️ Файл конфигурации не найден: `configs/stress_scenarios.yaml`"))

## 3️⃣ Детальный анализ результатов

После запуска стресс-теста здесь можно просмотреть детальные результаты.

In [ ]:
# Функция для детального анализа результатов
def show_detailed_results(scenario_name: str):
    """Показать детальные результаты сценария"""
    if scenario_name not in stress_results:
        print(f"⚠️ Результаты для сценария '{scenario_name}' не найдены")
        return
    
    result = stress_results[scenario_name]
    
    if not result.get("success"):
        print(f"❌ Сценарий '{scenario_name}' завершился с ошибкой: {result.get('error')}")
        return
    
    results = result["results"]
    if not results:
        print("⚠️ Результаты отсутствуют")
        return
    
    display(Markdown(f"## 📊 Детальные результаты: {result['scenario_label']}"))
    
    # Income Statement
    if "income_statement" in results:
        display(Markdown("### Income Statement"))
        display(results["income_statement"])
    
    # Balance Sheet
    if "balance_sheet" in results:
        display(Markdown("### Balance Sheet"))
        display(results["balance_sheet"])
    
    # Cash Flow
    if "cash_flow" in results:
        display(Markdown("### Cash Flow"))
        display(results["cash_flow"])

# Пример использования:
# show_detailed_results("mild")